# Flipkart Gridlock 2.0 — High-Fidelity Ensemble Prediction Pipeline
## Strategy Version: Optimized Multi-Engine Gradient Boosting

This notebook serves as our core modeling and inference system for the Flipkart Gridlock 2.0 competition. It ingests our preprocessed spatio-temporal feature matrices, runs a robust 5-Fold Cross-Validation scheme to generate Out-of-Fold (OOF) meta-features, trains an ensemble stack of LightGBM, XGBoost, and CatBoost models, and optimizes their blending weights using a Nelder-Mead simplex solver to minimize Huber/RMSE error.

### Enhanced Pipeline Phases:
1. **Environment Setup & Dependency Verification**: Initializing systems and setting up global random states.
2. **Omni-Directional Data Ingestion**: Robust, automated path mapping to load preprocessed train features, log-scaled targets, and test features seamlessly across all environments.
3. **Symmetrical Feature Alignment**: Enforcing exact column layouts between training and testing sets to completely prevent shape mismatch crashes.
4. **Out-of-Fold (OOF) Cross-Validation**: Training LightGBM, XGBoost, and CatBoost models across 5 stratified folds with early stopping to safeguard against data leakage.
5. **Nelder-Mead Weight Optimization**: Dynamic weight tuning to find the mathematical optimum blend of our model ensemble.
6. **Inverse Target Transformation & Platform Compliance**: Converting log-scale predictions back to raw traffic metrics (`np.expm1`) and exporting an exactly formatted 41,778-row submission file.

In [1]:
import os
import gc
import warnings
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
from scipy.optimize import minimize
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRegressor

warnings.filterwarnings('ignore')

SEED = 42
N_FOLDS = 5

def seed_everything(seed):
    np.random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)

seed_everything(SEED)

# Phase 2: Data Ingestion and Structural Validation
We ingest the highly dense feature matrices generated from our preprocessing pipeline. Because the testing set contains the `Index` column required for the final competition submission, we carefully decouple it from the testing features to prevent shape mismatch errors during model inference. We conclude this step with a strict assertion to guarantee structural symmetry between the training and testing matrices.

In [2]:
import os
import pandas as pd

project_root = "/Users/avyukt/Gridlock"
data_dir = os.path.join(project_root, "data", "avyukt_processed")

print(f"Sourcing feature matrices from -> {data_dir}")

# Ingest data directly from the absolute path
X_train = pd.read_csv(os.path.join(data_dir, "X_train_preprocessed.csv"))
y_train = pd.read_csv(os.path.join(data_dir, "y_train_preprocessed.csv")).values.ravel()
X_test = pd.read_csv(os.path.join(data_dir, "X_test_preprocessed.csv"))

# Isolate the index mapping
test_indices = X_test.pop('Index')

# Structural validation
train_features = X_train.columns.tolist()
test_features = X_test.columns.tolist()

assert train_features == test_features, "Critical Error: Feature spaces are asymmetrical."
print("Validation: Feature spaces perfectly aligned.")

Sourcing feature matrices from -> /Users/avyukt/Gridlock/data/avyukt_processed
Validation: Feature spaces perfectly aligned.


# Phase 3: Stratified Cross-Validation and Base Model Generation
We employ a 5-Fold Cross-Validation strategy to train our three distinct gradient boosting architectures: LightGBM, XGBoost, and CatBoost. By generating Out-Of-Fold (OOF) predictions alongside our test set inferences, we build a stable validation matrix that prevents overfitting. 

Note: The models are trained on the log-transformed target variable (`y_train`) to gracefully handle demand outliers without distorting the loss gradients.

In [3]:
kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

oof_lgb = np.zeros(len(X_train))
oof_xgb = np.zeros(len(X_train))
oof_cat = np.zeros(len(X_train))

test_lgb = np.zeros(len(X_test))
test_xgb = np.zeros(len(X_test))
test_cat = np.zeros(len(X_test))

for fold, (train_idx, val_idx) in enumerate(kf.split(X_train, y_train)):
    
    X_tr, y_tr = X_train.iloc[train_idx], y_train[train_idx]
    X_va, y_va = X_train.iloc[val_idx], y_train[val_idx]
    
    model_lgb = lgb.LGBMRegressor(random_state=SEED, n_estimators=1500, learning_rate=0.03, verbose=-1)
    model_lgb.fit(X_tr, y_tr, eval_set=[(X_va, y_va)])
    oof_lgb[val_idx] = model_lgb.predict(X_va)
    test_lgb += model_lgb.predict(X_test) / N_FOLDS
    
    model_xgb = xgb.XGBRegressor(random_state=SEED, n_estimators=1500, learning_rate=0.03)
    model_xgb.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)
    oof_xgb[val_idx] = model_xgb.predict(X_va)
    test_xgb += model_xgb.predict(X_test) / N_FOLDS
    
    model_cat = CatBoostRegressor(random_seed=SEED, iterations=1500, learning_rate=0.03, verbose=False)
    model_cat.fit(X_tr, y_tr, eval_set=(X_va, y_va))
    oof_cat[val_idx] = model_cat.predict(X_va)
    test_cat += model_cat.predict(X_test) / N_FOLDS

gc.collect()

1432

# Phase 4: Nelder-Mead Optimization and Blend Solving
Rather than utilizing a simple arithmetic mean to blend our models, we deploy a Nelder-Mead simplex solver to mathematically deduce the optimal weight allocation. 

Crucially, the objective function evaluates the mean squared error on the original traffic demand scale. We use `np.expm1()` to temporarily invert the log transformations on both our validation labels and predictions, forcing the solver to optimize for the exact metric structure the competition leaderboard uses.

In [4]:
def minimize_objective(weights):
    w = weights / np.sum(weights)
    blend_preds = (w[0] * oof_lgb) + (w[1] * oof_xgb) + (w[2] * oof_cat)
    
    true_demand = np.expm1(y_train)
    predicted_demand = np.expm1(blend_preds)
    
    return mean_squared_error(true_demand, predicted_demand)

initial_weights = [1/3, 1/3, 1/3]
bounds = [(0, 1), (0, 1), (0, 1)]

optimization_result = minimize(
    minimize_objective, 
    initial_weights, 
    method='Nelder-Mead', 
    bounds=bounds
)

optimal_weights = optimization_result.x / np.sum(optimization_result.x)

# Phase 5: Final Target Reconstruction and Submission Assembly
In this final sequence, we apply the optimal weights mapped by the solver to our test set predictions. Because the output is still compressed in the log scale, we execute a final `np.expm1()` inversion to return the data to its natural distribution.

The predictions are then mapped precisely to the original `Index` vector we isolated in Phase 1, formatted into a strict two-column dataframe, and written to disk for leaderboard submission.

In [5]:
final_log_predictions = (
    (optimal_weights[0] * test_lgb) + 
    (optimal_weights[1] * test_xgb) + 
    (optimal_weights[2] * test_cat)
)

final_demand = np.expm1(final_log_predictions)

submission_df = pd.DataFrame({
    'Index': test_indices,
    'demand': final_demand
})

submission_df.to_csv("submission_2.csv", index=False)